In [1]:
import pandas as pd
import re, html
import json

In [2]:
posts = pd.read_csv('../Corpus_data/filtered_posts.csv')

In [3]:
comments = pd.read_csv('../Corpus_data/filtered_comments.csv')

In [4]:
comments.loc[comments['author'] == '[deleted]', 'author'] = 'Deleted author'

In [5]:
comments_grouped = (
    comments
    .groupby("link_id")
    .apply(lambda df: "\n\n".join(
        f"Comment from {author}: {body}"
        for author, body in zip(df["author"], df["body"])
        if pd.notna(body)
    ))
    .reset_index(name="body")
)

/tmp/ipykernel_6441/3014014717.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: "\n\n".join(


In [6]:
docs = posts[['created_utc', 'id', 'title', 'selftext']]

In [7]:
docs['postdoc'] = 'Title: ' + docs['title'] + '\n\n' + 'Post: ' + docs['selftext']

/tmp/ipykernel_6441/1510089497.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs['postdoc'] = 'Title: ' + docs['title'] + '\n\n' + 'Post: ' + docs['selftext']


In [8]:
docs = docs.merge(
    comments_grouped,
    left_on="id",
    right_on="link_id",
    how="left"
)

In [9]:
docs['body'] = '\n\n' + docs['body']

In [10]:
docs["postdoc"] = (
    docs["postdoc"].fillna("") + 
    docs["body"].fillna("")
)

In [11]:
cleandocs = docs[['created_utc', 'id', 'postdoc']]

In [12]:
URL_RE = re.compile(r"""(?xi)
\b(
  https?://[^\s<>"'\]]+ |
  www\.[^\s<>"'\]]+
)\b
""")
MD_LINK_RE = re.compile(r"""\[([^\]]+)\]\((https?://[^)]+)\)""")
ZW_CHARS_RE = re.compile(r"[\u200B\u200C\u200D\u2060\uFEFF]")

EMOTICON_RE = re.compile(r"""
(?:
  [:;=8]
  [\-o\*']?
  [\)\]\(\[dDpP/\:\}\{@\|\\]
)
""", re.VERBOSE)

EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "]+",
    flags=re.UNICODE
)

In [13]:
def clean_text_keep_paragraphs(text: str) -> str:
    if not isinstance(text, str):
        return ""

    t = html.unescape(html.unescape(text))
    t = ZW_CHARS_RE.sub("", t)
    t = MD_LINK_RE.sub(r"\1", t)
    t = URL_RE.sub("", t)
    t = t.replace("*", "")
    t = EMOJI_RE.sub("", t)
    t = EMOTICON_RE.sub("", t)

    # Normalize spaces but preserve paragraph breaks
    t = re.sub(r"[ \t]+", " ", t)

    # Remove zero-width junk lines like &amp;#x200B; remnants
    t = re.sub(r"\n\s*\n\s*\n+", "\n\n", t)  # collapse 3+ blank lines to 2

    return t.strip()

In [14]:
cleandocs['postdoc'] = cleandocs['postdoc'].apply(clean_text_keep_paragraphs)

/tmp/ipykernel_6441/1292857723.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleandocs['postdoc'] = cleandocs['postdoc'].apply(clean_text_keep_paragraphs)


In [15]:
cleandocs.rename(columns={"postdoc": "text"})[["id", "text"]].to_json(
    "../Corpus_data/topicgpt_docs.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)